In [ ]:
# 실습 준비 — 11주차 통계적 가설검정
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["ch11_potato.csv", "ch11_training_rel.csv", "ch11_training_ind.csv", "ch11_ad.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 통계적 가설검정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# dir = '{경로}' # 데이터 폴더 우클릭 후 경로 복사하여 붙여넣기.
dir = '/content/drive/MyDrive/데이터통계분석/source/data'
os.chdir(dir)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

%precision 3
np.random.seed(1111)

In [ ]:
df = pd.read_csv('ch11_potato.csv')
sample = np.array(df['무게'])
sample

In [ ]:
s_mean = np.mean(sample)
s_mean

## 통계적 가설검정이란

### 통계적 가설검정의 흐름

In [ ]:
rv = stats.norm(130, np.sqrt(9/14))
rv.isf(0.95)

In [ ]:
z = (s_mean - 130) / np.sqrt(9/14)
z

In [ ]:
rv = stats.norm()
rv.isf(0.95)

In [ ]:
rv.cdf(z)

### 단측검정과 양측검정

In [ ]:
z = (s_mean - 130) / np.sqrt(9/14)
z

In [ ]:
rv = stats.norm()
rv.interval(0.95)

In [ ]:
rv.cdf(z) * 2

In [ ]:
rv = stats.norm(130, 3)

In [ ]:
c = stats.norm().isf(0.95)
n_samples = 10000
cnt = 0
for _ in range(n_samples):
    sample_ = np.round(rv.rvs(14), 2)
    s_mean_ = np.mean(sample_)
    z = (s_mean_ - 130) / np.sqrt(9/14)
    if z < c:
        cnt += 1
cnt / n_samples

In [ ]:
rv = stats.norm(128, 3)

In [ ]:
c = stats.norm().isf(0.95)
n_samples = 10000
cnt = 0
for _ in range(n_samples):
    sample_ = np.round(rv.rvs(14), 2)
    s_mean_ = np.mean(sample_)
    z = (s_mean_ - 130) / np.sqrt(9/14)
    if z >= c:
        cnt += 1

cnt / n_samples

## 가설검정

### 정규분포의 모평균에 대한 검정(모분산을 알고 있음)

In [ ]:
def pmean_test(sample, mean0, p_var, alpha=0.05):
    s_mean = np.mean(sample)
    n = len(sample)
    rv = stats.norm()
    interval = rv.interval(1-alpha)

    z = (s_mean - mean0) / np.sqrt(p_var/n)
    if interval[0] <= z <= interval[1]:
        print('귀무가설을 채택')
    else:
        print('귀무가설을 기각')

    if z < 0:
        p = rv.cdf(z) * 2
    else:
        p = (1 - rv.cdf(z)) * 2
    print(f'p값은 {p:.3f}')

In [ ]:
pmean_test(sample, 130, 9)

### 정규분포의 모분산에 대한 검정

In [ ]:
def pvar_test(sample, var0, alpha=0.05):
    u_var = np.var(sample, ddof=1)
    n = len(sample)
    rv = stats.chi2(df=n-1)
    interval = rv.interval(1-alpha)

    y = (n-1) * u_var / var0
    if interval[0] <= y <= interval[1]:
        print('귀무가설을 채택')
    else:
        print('귀무가설을 기각')

    if y < rv.isf(0.5):
        p = rv.cdf(y) * 2
    else:
        p = (1 - rv.cdf(y)) * 2
    print(f'p값은 {p:.3f}')

In [ ]:
pvar_test(sample, 9)

### 정규분포의 모평균에 대한 검정(모분산을 알지 못함)

In [ ]:
def pmean_test(sample, mean0, alpha=0.05):
    s_mean = np.mean(sample)
    u_var = np.var(sample, ddof=1)
    n = len(sample)
    rv = stats.t(df=n-1)
    interval = rv.interval(1-alpha)

    t = (s_mean - mean0) / np.sqrt(u_var/n)
    if interval[0] <= t <= interval[1]:
        print('귀무가설을 채택')
    else:
        print('귀무가설을 기각')

    if t < 0:
        p = rv.cdf(t) * 2
    else:
        p = (1 - rv.cdf(t)) * 2
    print(f'p값은 {p:.3f}')

In [ ]:
pmean_test(sample, 130)

In [ ]:
t, p = stats.ttest_1samp(sample, 130)
t, p

## 2표본 문제에 관한 가설검정

### 대응비교 t검정

In [ ]:
training_rel = pd.read_csv('ch11_training_rel.csv')
print(training_rel.shape)
training_rel.head()

In [ ]:
training_rel['차'] = training_rel['후'] - training_rel['전']
training_rel.head()

In [ ]:
t, p = stats.ttest_1samp(training_rel['차'], 0)
p

In [ ]:
t, p = stats.ttest_rel(training_rel['후'], training_rel['전'])
p

### 독립비교 t검정

In [ ]:
training_ind = pd.read_csv('ch11_training_ind.csv')
print(training_ind.shape)
training_ind.head()

In [ ]:
t, p = stats.ttest_ind(training_ind['A'], training_ind['B'],
                       equal_var=False)
p

### 윌콕슨의 부호순위검정

In [ ]:
training_rel = pd.read_csv('ch11_training_rel.csv')
toy_df = training_rel[:6].copy()
toy_df

In [ ]:
diff = toy_df['후'] - toy_df['전']
toy_df['차'] = diff
toy_df

In [ ]:
rank = stats.rankdata(abs(diff)).astype(int)
toy_df['순위'] = rank
toy_df

In [ ]:
r_minus = np.sum((diff < 0) * rank)
r_plus = np.sum((diff > 0) * rank)

r_minus, r_plus

In [ ]:
toy_df['후'] = toy_df['전'] + np.arange(1, 7)
diff = toy_df['후'] - toy_df['전']
rank = stats.rankdata(abs(diff)).astype(int)
toy_df['차'] = diff
toy_df['순위'] = rank
toy_df

In [ ]:
r_minus = np.sum((diff < 0) * rank)
r_plus = np.sum((diff > 0) * rank)

r_minus, r_plus

In [ ]:
toy_df['후'] = toy_df['전'] + [1, -2, -3, 4, 5, -6]
diff = toy_df['후'] - toy_df['전']
rank = stats.rankdata(abs(diff)).astype(int)
toy_df['차'] = diff
toy_df['순위'] = rank
toy_df

In [ ]:
r_minus = np.sum((diff < 0) * rank)
r_plus = np.sum((diff > 0) * rank)

r_minus, r_plus

In [ ]:
T, p = stats.wilcoxon(training_rel['전'], training_rel['후'])
p

In [ ]:
T, p = stats.wilcoxon(training_rel['후'] - training_rel['전'])
p

In [ ]:
n = 10000
diffs = np.round(stats.norm(3, 4).rvs(size=(n, 20)))

In [ ]:
cnt = 0
alpha = 0.05
for diff in diffs:
    t, p = stats.ttest_1samp(diff, 0)
    if p < alpha:
        cnt += 1
cnt / n

In [ ]:
cnt = 0
alpha = 0.05
for diff in diffs:
    T, p = stats.wilcoxon(diff)
    if p < alpha:
        cnt += 1
cnt / n

### 만・위트니의 U검정

In [ ]:
training_ind = pd.read_csv('ch11_training_ind.csv')
toy_df = training_ind[:5].copy()
toy_df

In [ ]:
rank = stats.rankdata(np.concatenate([toy_df['A'],
                                      toy_df['B']]))
rank_df = pd.DataFrame({'A': rank[:5],
                        'B': rank[5:10]}).astype(int)
rank_df

In [ ]:
n1 = len(rank_df['A'])
u = rank_df['A'].sum() - (n1*(n1+1))/2
u

In [ ]:
rank_df = pd.DataFrame(np.arange(1, 11).reshape(2, 5).T,
                       columns=['A', 'B'])
rank_df

In [ ]:
u = rank_df['A'].sum() - (n1*(n1+1))/2
u

In [ ]:
rank_df = pd.DataFrame(np.arange(1, 11).reshape(2, 5)[::-1].T,
                       columns=['A', 'B'])
rank_df

In [ ]:
u = rank_df['A'].sum() - (n1*(n1+1))/2
u

In [ ]:
u, p = stats.mannwhitneyu(training_ind['A'], training_ind['B'],
                          alternative='two-sided')
p

### 카이제곱검정

In [ ]:
ad_df = pd.read_csv('ch11_ad.csv')
n = len(ad_df)
print(n)
ad_df.head()

In [ ]:
ad_cross = pd.crosstab(ad_df['광고'], ad_df['구입'])
ad_cross

In [ ]:
ad_cross['했다'] / (ad_cross['했다'] + ad_cross['하지 않았다'])

In [ ]:
n_not, n_yes = ad_cross.sum()
n_not, n_yes

In [ ]:
n_adA, n_adB = ad_cross.sum(axis=1)
n_adA, n_adB

In [ ]:
ad_ef = pd.DataFrame({'했다': [n_adA * n_yes / n,
                              n_adB * n_yes / n],
                      '하지 않았다': [n_adA * n_not / n,
                                   n_adB * n_not / n]},
                      index=['A', 'B'])
ad_ef

In [ ]:
y = ((ad_cross - ad_ef) ** 2 / ad_ef).sum().sum()
y

In [ ]:
rv = stats.chi2(1)
1 - rv.cdf(y)

In [ ]:
chi2, p, dof, ef = stats.chi2_contingency(ad_cross,
                                          correction=False)
chi2, p, dof

In [ ]:
ef